# Análisis Exploratorio y Proyección de Comportamiento Comercial
Dataset: Sample Superstore — ventas retail por periodo, categoría y producto.  
Objetivo: identificar tendencias, detectar anomalías y proyectar la demanda futura.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Carga y revisión inicial

In [ ]:
df = pd.read_csv('../data/raw/superstore.csv', encoding='latin-1')

print(f'Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Limpieza y preparación

In [ ]:
# Nulos por columna
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else 'Sin valores nulos')

In [ ]:
# Duplicados
dupes = df.duplicated().sum()
print(f'Filas duplicadas: {dupes}')
if dupes > 0:
    df.drop_duplicates(inplace=True)

In [ ]:
# Convertir fechas y crear columnas de tiempo
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

df['year']       = df['Order Date'].dt.year
df['month']      = df['Order Date'].dt.month
df['year_month'] = df['Order Date'].dt.to_period('M')

# Estandarizar texto
for col in ['Segment', 'Region', 'Category', 'Sub-Category', 'Ship Mode']:
    df[col] = df[col].str.strip()

print('Rango de fechas:', df['Order Date'].min().date(), '→', df['Order Date'].max().date())

## 3. Detección de valores atípicos

In [ ]:
# Método IQR sobre Sales y Profit
for col in ['Sales', 'Profit']:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f'{col}: {len(outliers)} outliers  |  rango normal [{lower:.2f}, {upper:.2f}]')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ['Sales', 'Profit']):
    sns.boxplot(y=df[col], ax=ax, color='steelblue', flierprops=dict(marker='o', markersize=3))
    ax.set_title(f'Distribución de {col}')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('../images/01_outliers_boxplot.png')
plt.show()

## 4. Análisis por periodo

In [ ]:
monthly = (
    df.groupby('year_month')
    .agg(sales=('Sales', 'sum'), profit=('Profit', 'sum'), orders=('Order ID', 'nunique'))
    .reset_index()
)
monthly['year_month_str'] = monthly['year_month'].astype(str)
monthly.tail(6)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(monthly['year_month_str'], monthly['sales'], color='steelblue', linewidth=1.8, label='Ingresos')
ax.plot(monthly['year_month_str'], monthly['profit'], color='seagreen', linewidth=1.8, linestyle='--', label='Beneficio')

tick_step = max(1, len(monthly) // 12)
ax.set_xticks(monthly['year_month_str'][::tick_step])
ax.set_xticklabels(monthly['year_month_str'][::tick_step], rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title('Ingresos y Beneficio mensual')
ax.legend()

plt.tight_layout()
plt.savefig('../images/02_monthly_trend.png')
plt.show()

## 5. Análisis por categoría y subcategoría

In [ ]:
by_cat = (
    df.groupby(['Category', 'Sub-Category'])
    .agg(sales=('Sales', 'sum'), profit=('Profit', 'sum'))
    .reset_index()
    .sort_values('sales', ascending=False)
)
by_cat['margin_pct'] = (by_cat['profit'] / by_cat['sales'] * 100).round(2)
by_cat

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ingresos por subcategoría
sns.barplot(data=by_cat, y='Sub-Category', x='sales', hue='Category', ax=axes[0], dodge=False)
axes[0].set_title('Ingresos por subcategoría')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
axes[0].set_xlabel('')
axes[0].set_ylabel('')

# Margen por subcategoría
colors = ['tomato' if m < 0 else 'steelblue' for m in by_cat['margin_pct']]
axes[1].barh(by_cat['Sub-Category'], by_cat['margin_pct'], color=colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Margen % por subcategoría')
axes[1].set_xlabel('Margen %')

plt.tight_layout()
plt.savefig('../images/03_category_analysis.png')
plt.show()

## 6. Análisis por segmento y región

In [ ]:
by_segment = (
    df.groupby('Segment')
    .agg(sales=('Sales', 'sum'), profit=('Profit', 'sum'), orders=('Order ID', 'nunique'))
    .reset_index()
)

by_region = (
    df.groupby('Region')
    .agg(sales=('Sales', 'sum'), profit=('Profit', 'sum'))
    .reset_index()
    .sort_values('sales', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(data=by_segment, x='Segment', y='sales', ax=axes[0], palette='Blues_d')
axes[0].set_title('Ingresos por segmento')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
axes[0].set_xlabel('')

sns.barplot(data=by_region, x='Region', y='sales', ax=axes[1], palette='Greens_d')
axes[1].set_title('Ingresos por región')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
axes[1].set_xlabel('')

plt.tight_layout()
plt.savefig('../images/04_segment_region.png')
plt.show()

## 7. Impacto del descuento en el margen

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.scatterplot(data=df, x='Discount', y='Profit', alpha=0.3, hue='Category', ax=ax)
ax.axhline(0, color='red', linewidth=0.8, linestyle='--')
ax.set_title('Descuento vs Beneficio')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('../images/05_discount_vs_profit.png')
plt.show()

## 8. Regresión lineal — proyección de ingresos

In [ ]:
# Preparar serie mensual con índice numérico
monthly['t'] = np.arange(len(monthly))

X = monthly[['t']].values
y = monthly['sales'].values

model = LinearRegression()
model.fit(X, y)

y_pred = model.predict(X)

print(f'R²  : {r2_score(y, y_pred):.4f}')
print(f'MAE : ${mean_absolute_error(y, y_pred):,.2f}')
print(f'Tendencia mensual: ${model.coef_[0]:,.2f}')

In [ ]:
# Proyectar 6 meses hacia adelante
last_t    = monthly['t'].max()
future_t  = np.arange(last_t + 1, last_t + 7).reshape(-1, 1)
future_sales = model.predict(future_t)

last_date    = df['Order Date'].max()
future_dates = pd.date_range(start=last_date + pd.offsets.MonthBegin(1), periods=6, freq='MS')

forecast_df = pd.DataFrame({'date': future_dates, 'sales_forecast': future_sales})
print(forecast_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(monthly['year_month_str'], y, color='steelblue', linewidth=1.5, label='Histórico')
ax.plot(monthly['year_month_str'], y_pred, color='orange', linewidth=1.5, linestyle='--', label='Tendencia (regresión)')

future_labels = forecast_df['date'].dt.strftime('%Y-%m').tolist()
ax.plot(
    list(range(len(monthly), len(monthly) + 6)),
    future_sales,
    color='tomato', linewidth=1.5, linestyle=':', marker='o', markersize=4, label='Proyección'
)

tick_step = max(1, len(monthly) // 12)
ax.set_xticks(range(0, len(monthly), tick_step))
ax.set_xticklabels(monthly['year_month_str'][::tick_step], rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title('Proyección de ingresos — Regresión lineal')
ax.legend()

plt.tight_layout()
plt.savefig('../images/06_linear_forecast.png')
plt.show()

## 9. Serie temporal — promedio móvil y descomposición de tendencia

In [ ]:
# Promedio móvil de 3 y 6 meses para suavizar estacionalidad
monthly['ma3'] = monthly['sales'].rolling(window=3, center=True).mean()
monthly['ma6'] = monthly['sales'].rolling(window=6, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(monthly['year_month_str'], monthly['sales'], color='steelblue', linewidth=1.2, alpha=0.6, label='Ventas mensuales')
ax.plot(monthly['year_month_str'], monthly['ma3'],   color='orange',    linewidth=1.8, label='Media móvil 3M')
ax.plot(monthly['year_month_str'], monthly['ma6'],   color='seagreen',  linewidth=1.8, label='Media móvil 6M')

tick_step = max(1, len(monthly) // 12)
ax.set_xticks(monthly['year_month_str'][::tick_step])
ax.set_xticklabels(monthly['year_month_str'][::tick_step], rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title('Serie temporal con promedios móviles')
ax.legend()

plt.tight_layout()
plt.savefig('../images/07_moving_average.png')
plt.show()

In [ ]:
# Variación mensual (MoM) para detectar picos y caídas
monthly['mom_pct'] = monthly['sales'].pct_change() * 100

fig, ax = plt.subplots(figsize=(14, 3))
colors = ['tomato' if v < 0 else 'steelblue' for v in monthly['mom_pct'].fillna(0)]
ax.bar(monthly['year_month_str'], monthly['mom_pct'].fillna(0), color=colors)
ax.axhline(0, color='black', linewidth=0.8)

tick_step = max(1, len(monthly) // 12)
ax.set_xticks(monthly['year_month_str'][::tick_step])
ax.set_xticklabels(monthly['year_month_str'][::tick_step], rotation=45, ha='right', fontsize=8)
ax.set_title('Variación mensual de ingresos (MoM %)')
ax.set_ylabel('%')

plt.tight_layout()
plt.savefig('../images/08_mom_variation.png')
plt.show()

## 10. Exportar dataset procesado

In [ ]:
df.to_csv('../data/processed/superstore_clean.csv', index=False)
monthly.to_csv('../data/processed/monthly_summary.csv', index=False)
forecast_df.to_csv('../data/processed/sales_forecast.csv', index=False)

print('Archivos exportados a data/processed/')